In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from pydrake.all import (DiagramBuilder, LeafSystem, Meshcat, MeshcatVisualizer,
                         MeshcatVisualizerParams, PiecewisePolynomial,
                         PiecewisePose, PiecewiseQuaternionSlerp,
                         RigidTransform, RotationMatrix, Simulator,
                         TrajectorySource, Value, plot_system_graphviz)
from shoelace_tying_station import ShoelaceTyingStation
IS_TEST = "BAZEL_TEST" in os.environ

In [ ]:
meshcat = Meshcat()
meshcat.SetCameraPose([0.08, -0.02, 0.23], [-0.08, 0, 0.085])
meshcat.SetProperty("/Grid", "visible", False)
meshcat.SetProperty("/Axes", "visible", False)

In [ ]:
class PoseTrajectorySource(LeafSystem):
    def __init__(self, times, poses):
        super().__init__()
        self._traj = PiecewisePose(
            PiecewisePolynomial.FirstOrderHold(times, np.array([pose.translation() for pose in poses]).T),
            PiecewiseQuaternionSlerp(times, [pose.rotation() for pose in poses])
        )
        self.DeclareAbstractOutputPort(
            name="y0",
            alloc=lambda: Value(RigidTransform()),
            calc=self.CalcOutput
        )

    def CalcOutput(self, context, output):
        t = context.get_time()
        output.set_value(self._traj.GetPose(t))

In [ ]:
builder = DiagramBuilder()

# Add manipulation station
station = ShoelaceTyingStation(
    plant_dt=0.8e-3,
    lace_as_capsule_chain=False,
)
builder.AddSystem(station)
X_WH_left_init, X_WH_right_init = station.GetDefaultHandPoses()

# Time stamps of trajectory
times = [0, 1.0, 1.4, 2.0, 3.4, 3.8, 4.4, 4.6, 4.8, 5.0, 6.0, 6.2, 7.0]

# Left hand pose trajectory
R_WH_left1 = RotationMatrix(np.array([[0,1,0], [0,0,-1], [-1,0,0]]))
R_WH_left2 = RotationMatrix(np.array([[1,0,0], [0,0,-1], [0,1,0]]))
lh_pose_traj_source = PoseTrajectorySource(
    times,
    [
        X_WH_left_init,
        RigidTransform(R_WH_left1, [-0.03, 0.20, 0.065]),
        RigidTransform(R_WH_left1, [-0.03, 0.16, 0.065]),
        RigidTransform(R_WH_left1, [0.03, 0.14, 0.065]),
        RigidTransform(R_WH_left2, [0, 0.14, 0.183]),
        RigidTransform(R_WH_left2, [-0.01, 0.14, 0.183]),
        RigidTransform(R_WH_left2, [-0.01, 0.105, 0.183]),
        RigidTransform(R_WH_left2, [-0.01, 0.105, 0.183]),
        RigidTransform(R_WH_left2, [-0.013, 0.105, 0.183]),
        RigidTransform(R_WH_left2, [-0.013, 0.105, 0.183]),
        RigidTransform(R_WH_left2, [0, 0.22, 0.11]),
        RigidTransform(R_WH_left2, [0, 0.22, 0.11]),
        X_WH_left_init,
    ]
)
builder.AddSystem(lh_pose_traj_source)
builder.Connect(lh_pose_traj_source.get_output_port(), station.GetInputPort("left_hand_pose"))

# Left hand grasp trajectory
lh_grasp_traj_source = TrajectorySource(
    PiecewisePolynomial.FirstOrderHold(
        times,
        [
            [0.95] * 5 +
            [0.65] * 4 +
            [1.0] * 2 +
            [0.0] * 2
        ]
))
builder.AddSystem(lh_grasp_traj_source)
builder.Connect(lh_grasp_traj_source.get_output_port(), station.GetInputPort("left_hand_grasp"))

# Right hand pose trajectory
R_WH_right1 = RotationMatrix(np.array([[0,-1,0], [0,0,1], [-1,0,0]]))
rh_pose_traj_source = PoseTrajectorySource(
    times,
    [
        X_WH_right_init,
        RigidTransform(R_WH_right1, [0.03, -0.20, 0.065]),
        RigidTransform(R_WH_right1, [0.03, -0.15, 0.065]),
        RigidTransform(R_WH_right1, [-0.03, -0.143, 0.065]),
        RigidTransform(R_WH_right1, [0, -0.14, 0.18]),
        RigidTransform(R_WH_right1, [0.016, -0.14, 0.18]),
        RigidTransform(R_WH_right1, [0.016, -0.093, 0.18]),
        RigidTransform(R_WH_right1, [0.022, -0.093, 0.18]),
        RigidTransform(R_WH_right1, [0.022, -0.093, 0.18]),
        RigidTransform(R_WH_right1, [0.022, -0.09, 0.18]),
        RigidTransform(R_WH_right1, [0, -0.22, 0.11]),
        RigidTransform(R_WH_right1, [0, -0.22, 0.11]),
        X_WH_right_init,
    ]
)
builder.AddSystem(rh_pose_traj_source)
builder.Connect(rh_pose_traj_source.get_output_port(), station.GetInputPort("right_hand_pose"))

# Right hand grasp trajectory
rh_grasp_traj_source = TrajectorySource(
    PiecewisePolynomial.FirstOrderHold(
        times,
        [
            [0.95] * 5 +
            [0.65] * 2 +
            [1.0] * 4 +
            [0.0] * 2
        ]
))
builder.AddSystem(rh_grasp_traj_source)
builder.Connect(rh_grasp_traj_source.get_output_port(), station.GetInputPort("right_hand_grasp"))

# Add Meshcat visualizer
visualizer = MeshcatVisualizer.AddToBuilder(
    builder, station.GetOutputPort("scene_graph_query"), meshcat,
    MeshcatVisualizerParams(publish_period=1/300)
)

# Build diagram
diagram = builder.Build()
plot_system_graphviz(diagram)

In [ ]:
simulator = Simulator(diagram)
simulator.Initialize()

elastic_energy = []
def monitor(context):
    station_context = diagram.GetSubsystemContext(station, context)
    elastic_energy.append(
        (context.get_time(), station.GetElasticEnergy(station_context))
    )
simulator.set_monitor(monitor)

visualizer.StartRecording()
simulator.AdvanceTo(times[-1] + 1e-5 if not IS_TEST else 2.0)
visualizer.StopRecording()
visualizer.PublishRecording()